# Perform inference with EquiFusion

Given a sequence of 3D joint positions representing the past (0.5 s), here we run EquiFusion to predict corresponding future motions (2 s)!

EquiFusion is permutation equivariant, so a single model can be applied to **different skeletons** (different numbers of joints and kinematic trees).
Here we predict the motion of an input having the AMASS skeleton, and then apply the *same* model zero-shot to a motion parametrized with the H36M skeleton, which it has never seen during training.

## Select model and data
We use EquiFusion A+N, the model trained on AMASS and Nymeria jointly (`cross_kinematics/amass_nymeria`). Other released models are `cross_kinematics/{amass_nymeria_h36m, amass_nymeria_3d, amass}` and the in-domain models `in_domain/{amass, nymeria, h36m}`.
The example below is an observation from the AMASS test set (30 frames at 60 fps, joint positions in meters, root joint included).

In [1]:
# Released models, expected in ./trained_models; they can also be downloaded from Hugging Face, e.g.
# checkpoint_path = download_equifusion('EquiFusion/equifusion-cross-kinematics-amass-nymeria')  (from src.inference_utils)
model_name = 'cross_kinematics/amass_nymeria'
checkpoint_path = f'./trained_models/{model_name}/checkpoints/checkpoint_375.safetensors'
dataset_name = 'amass'  # skeleton of the input data: 'amass', 'nymeria' or 'h36m'
num_samples = 50

## Load the model

In [2]:
import numpy as np
import torch
from src.inference_utils import load_equifusion, predict

model, skeleton, config = load_equifusion(checkpoint_path, dataset_name, num_samples=num_samples)

Created encoder with 2.023M and decoder with 3.471M trainable parameters.


Model size Autoencoder (MB):  20.9599609375
Diffusion is_ddim_sampling:  False
Model size Diffusion (MB):  9.9093017578125


## Load the example or use your own
`obs` has shape `[T_obs, J, 3]` (or `[B, T_obs, J, 3]`): joint positions in meters in the joint order of the dataset, including the root joint. The input frame rate must match the dataset (60 fps for AMASS and Nymeria, 50 fps for H36M, in both cases 0.5 s of observation).
The prediction has shape `[B, num_samples, T_pred, J-1, 3]`: the joints without the root, relative to the root.

In [3]:
# input: joint positions in meters with the root (hip) joint included; as in SkeletonDiffusion, the root is at the origin in every frame
obs = np.load('figures/examples/amass_inference_obs.npy')  # [30, 22, 3], AMASS test sequence
gt = np.load('figures/examples/amass_inference_gt.npy')    # [120, 22, 3], ground-truth future (only used for visualization)
print(obs.shape, gt.shape)

(30, 22, 3) (120, 22, 3)


## Predict

In [4]:
pred = predict(model, skeleton, config, obs, num_samples=num_samples, seed=0)  # [num_samples, T_pred, J-1, 3]
print(pred.shape)

torch.Size([50, 120, 21, 3])


## Rank the samples
Pick the sample closest to the ground truth, and the samples that are the most different from it and from each other (diversity ranking, as in SkeletonDiffusion).
Any other criterion can be used instead, e.g. metrics in `src.metrics`.

In [5]:
from src.metrics.ranking import get_closest_and_nfurthest_maxapd

pred = pred.cpu()
gt_rel = torch.from_numpy(gt[:, 1:] - gt[:, :1])  # ground truth relative to the root, as the prediction
ade = (pred - gt_rel).norm(dim=-1).mean(dim=(-1, -2))  # average displacement error of each sample [meters]
print(f'minADE over {num_samples} samples: {ade.min():.3f} m')

closest, furthest, sample_idxs = get_closest_and_nfurthest_maxapd(pred, gt_rel, nsamples=3)
selected = torch.cat([closest.unsqueeze(0), furthest]).numpy()

minADE over 50 samples: 0.107 m


## Visualize
The observed motion is shown in gray, followed by the ground truth (green), the closest sample (red) and three diverse samples (blue).

In [6]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML


def with_root(x):
    """[..., J-1, 3] predictions (relative to the root) -> [..., J, 3] with the root at the origin."""
    return np.concatenate([np.zeros_like(x[..., :1, :]), x], axis=-2)


def animate(obs, futures, titles, limbseqs, obs_limbseq, colors=None, step=2, fps=30):
    """Skeleton animation: the observed motion (gray) followed by one or more futures, one panel each.
    obs [T_obs, J, 3]; futures: list of [T_pred, J, 3], all root-relative with the root joint included (NaN = not drawn);
    limbseqs: list of bone lists (one per future); obs_limbseq: bones for the observed motion."""
    n, T_obs, T_pred = len(futures), len(obs), len(futures[0])
    colors = colors or ["tab:green"] + ["tab:red"] + ["tab:blue"] * n
    obs = obs - obs[:, :1]
    fig = plt.figure(figsize=(3 * n, 3.4), dpi=80)
    axes, lines = [], []
    for i in range(n):
        ax = fig.add_subplot(1, n, i + 1, projection="3d")
        ax.set(xlim=(-1, 1), ylim=(-1, 1), zlim=(-1, 1), title=titles[i])
        ax.set_box_aspect((1, 1, 1), zoom=1.35); ax.view_init(elev=10, azim=-60); ax.set_axis_off()
        lines.append([ax.plot([], [], [], lw=2)[0] for _ in range(max(len(limbseqs[i]), len(obs_limbseq)))])
    fig.subplots_adjust(left=0, right=1, bottom=0, top=0.9, wspace=0)

    def draw(t):
        for i in range(n):
            pose, bones, color = (obs[t], obs_limbseq, "gray") if t < T_obs else (futures[i][t - T_obs], limbseqs[i], colors[i])
            for k, line in enumerate(lines[i]):
                if k < len(bones):
                    a, b = bones[k]
                    line.set_data_3d(pose[[a, b], 0], pose[[a, b], 1], pose[[a, b], 2])
                else:
                    line.set_data_3d([], [], [])
                line.set_color(color)
        return sum(lines, [])

    anim = animation.FuncAnimation(fig, draw, frames=range(0, T_obs + T_pred, step), interval=1000 / fps)
    html = HTML(anim.to_html5_video())
    plt.close(fig)
    return html


In [7]:
animate(obs, [gt - gt[:, :1], *with_root(selected)],
        titles=['Ground truth', 'Closest to GT', 'Diverse 1', 'Diverse 2', 'Diverse 3'],
        limbseqs=[skeleton.limbseq] * 5, obs_limbseq=skeleton.limbseq)

# Zero-shot kinematics: a skeleton never seen during training
The model was trained on AMASS and Nymeria only and has never seen the H36M skeleton (17 joints, different kinematic tree, 25 frames at 50 fps for the observation). It still runs on it zero-shot: we only need to load it for `dataset_name='h36m'`.

In [8]:
model_h36m, skeleton_h36m, config_h36m = load_equifusion(checkpoint_path, 'h36m', num_samples=num_samples)

obs_h36m = np.load('figures/examples/h36m_obs.npy')  # [25, 17, 3], H36M test sequence (50 fps)
gt_h36m = np.load('figures/examples/h36m_gt.npy')    # [100, 17, 3]
pred_h36m = predict(model_h36m, skeleton_h36m, config_h36m, obs_h36m, num_samples=num_samples, seed=0).cpu()  # [50, 100, 16, 3]

gt_rel_h36m = torch.from_numpy(gt_h36m[:, 1:] - gt_h36m[:, :1])
print(f'H36M skeleton, {obs_h36m.shape[1]} joints. minADE: {(pred_h36m - gt_rel_h36m).norm(dim=-1).mean(dim=(-1, -2)).min():.3f} m')

closest, furthest, _ = get_closest_and_nfurthest_maxapd(pred_h36m, gt_rel_h36m, nsamples=3)
selected_h36m = torch.cat([closest.unsqueeze(0), furthest]).numpy()
animate(obs_h36m, [gt_h36m - gt_h36m[:, :1], *with_root(selected_h36m)],
        titles=['Ground truth', 'Closest to GT', 'Diverse 1', 'Diverse 2', 'Diverse 3'],
        limbseqs=[skeleton_h36m.limbseq] * 5, obs_limbseq=skeleton_h36m.limbseq, step=2, fps=25)

Created encoder with 2.023M and decoder with 3.471M trainable parameters.
Model size Autoencoder (MB):  20.9599609375
Diffusion is_ddim_sampling:  False
Model size Diffusion (MB):  9.9093017578125


H36M skeleton, 17 joints. minADE: 0.048 m
